# Mixed-layer depth — why `at MLD` comes out blotchy

Every depth subset emits an `_mld` channel, and in almost all of them
that row of the maps looks blotchy in a way the `_sfc`, `_z25m` and
`_mld_mean` rows do not.  This notebook works out why, and what the
alternatives cost.

| | |
|---|---|
| Domain | one 720 × 720 × 51 tile (Section 1) |
| Affects | every `_mld` and `_mld_mean` channel in the DEPTH pipeline |
| Current definition | `calculate_fields_at_depth.mixed_layer_depth` |

## The suspicion, and what is actually going on

The natural suspicion is that `_extract_at_mld` samples a single grid
cell without interpolating, and that interpolation would fix it.  The
first half is true and the second is misleading, and the distinction
decides what to change.

`mixed_layer_depth` returns **the deepest model level Z at which
σ₀ − σ₀(10 m) ≤ 0.03**.  So the MLD it returns is *already* a model
level, not a continuous depth.  Given that, `_extract_at_mld`'s
nearest-k lookup is **exact** — it recovers the level the definition
named, and adding interpolation to the extraction would change
nothing.

The blotchiness comes one step earlier: **MLD is a staircase.**  It can
only take the 51 discrete values in `Z`, and near typical Gulf Stream
mixed layers those are 10–20 m apart.  Two neighbouring columns whose
true mixed layer differs by a metre can therefore land on levels tens
of metres apart, and any field sampled there inherits the jump.  The
map is not noisy — it is quantised.

So the fix, if we want one, is to make **the MLD itself continuous**,
and only then to interpolate the field to it.  Sections 3–5 build that
and compare.


## Section 1 — Setup

In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0
# ------------------------------------------------------------------------

import dask
import numpy as np
import xarray as xr

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
from dbof.preprocessing import vertical_helpers as VH
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile

# tile_utils sets the Agg backend on import; restore inline afterwards.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)
tile = rect_ij_to_tile(
    *tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3))
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}")

ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(
    S3, DATE, tile, ["Theta", "Salt"])
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)

# Depth coordinate, positive downward, and the layer thicknesses.
Z = np.asarray(VH._get_depth_coord(ds_merge).values, dtype=float)
print(f"levels : {len(Z)}, {Z[0]:.1f} m to {Z[-1]:.1f} m")

## Section 2 — The staircase, shown directly

Two maps.  The MLD in metres, and the **model level index** it
corresponds to.  If the index map is piecewise-constant with sharp
boundaries, the staircase is the mechanism and nothing else needs to
be argued.


In [ ]:
mld = CFAD.mixed_layer_depth(ds_merge)
rho = CF.potential_density(ds_merge)

# The level index each column's MLD lands on.
k_of_mld = xr.apply_ufunc(
    lambda m: np.abs(Z[np.newaxis, :] - m.reshape(-1, 1)).argmin(axis=1)
                .reshape(m.shape).astype("float32"),
    mld, dask="parallelized", output_dtypes=[np.float32],
)

vals = dfig.pack_tile_levels(
    dict(zip(["mixed_layer_depth", "k_of_mld"],
             dask.compute(mld, k_of_mld))),
    XC, YC, edge_margin=0, land_mask=LAND, levels=("sfc",),
    verbose=False)

print("level spacing where the MLD lands:")
kk = vals["k_of_mld"]["sfc"][2]
for q in (5, 25, 50, 75, 95):
    k = int(np.nanpercentile(kk, q))
    print(f"  p{q:<3d}  k={k:2d}  z={Z[k]:7.1f} m   "
          f"gap to next level {Z[min(k + 1, len(Z) - 1)] - Z[k]:5.1f} m")
print(f"\ndistinct levels used across the tile: "
      f"{int(np.nanmax(kk) - np.nanmin(kk)) + 1}")

In [ ]:
dfig.depth_map_grid(
    ["mixed_layer_depth", "k_of_mld"], vals, CMAP_CFG,
    region=REGION, levels=("sfc",),
    row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 1 — MLD in metres, and the model level it snaps "
              "to.  The second panel is the staircase."))
plt.show()

## Section 3 — A continuous MLD

Same 0.03 kg m⁻³ criterion, but instead of returning the last level
that satisfies it, interpolate **linearly in z** between the bracketing
levels to find where the threshold is crossed exactly.  This is the
smallest possible change to the definition: same threshold, same
reference depth, continuous output.


In [ ]:
def mld_interpolated(ds_merge, threshold=0.03, ref_depth_m=10.0):
    """Continuous MLD: linear interpolation to the exact crossing.

    Same criterion as ``calculate_fields_at_depth.mixed_layer_depth``
    (sigma0 exceeding a reference value by *threshold*), but the depth
    is interpolated between the two bracketing model levels instead of
    being snapped to the deeper one.

    Inputs
    ------
    ds_merge : xr.Dataset
        Merged tile dataset.
    threshold : float
        Density criterion, kg m-3.
    ref_depth_m : float
        Reference depth for sigma0, m.

    Outputs
    -------
    xr.DataArray
        2D MLD in metres, positive downward, dask-backed.

    Generated by LH and Claude.
    """
    sigma0 = CF.potential_density_anomaly(ds_merge)
    zdim = VH._get_vertical_dim(sigma0)
    k_ref = int(np.abs(Z - ref_depth_m).argmin())

    def _interp(sig):
        # sig: (..., nk) -- apply_ufunc puts the vertical axis last.
        flat = sig.reshape(-1, sig.shape[-1]).astype(np.float64)
        excess = flat - (flat[:, k_ref][:, None] + threshold)
        out = np.full(flat.shape[0], Z[-1], dtype=np.float64)
        for n in range(flat.shape[0]):
            e = excess[n]
            idx = np.nonzero(e > 0)[0]
            idx = idx[idx > k_ref]
            if idx.size == 0:
                continue
            k = idx[0]
            e0, e1 = e[k - 1], e[k]
            # Linear crossing of excess == 0 between k-1 and k.
            out[n] = (Z[k - 1] + (Z[k] - Z[k - 1]) * (-e0) / (e1 - e0)
                      if e1 != e0 else Z[k])
        return out.reshape(sig.shape[:-1]).astype(np.float32)

    return xr.apply_ufunc(
        _interp, sigma0,
        input_core_dims=[[zdim]], dask="parallelized",
        output_dtypes=[np.float32],
    )


mld_i = mld_interpolated(ds_merge)
mld_di = CFAD.mixed_layer_depth_DI(ds_merge)

mlds = dict(zip(["mld_threshold", "mld_interp", "mld_DI"],
                dask.compute(mld, mld_i, mld_di)))
mv = dfig.pack_tile_levels(mlds, XC, YC, edge_margin=0,
                           land_mask=LAND, levels=("sfc",),
                           verbose=False)

print(f"{'definition':<16}{'median':>10}{'p5':>9}{'p95':>9}"
      f"{'distinct values':>18}")
print("-" * 62)
for nm in ("mld_threshold", "mld_interp", "mld_DI"):
    a = mv[nm]["sfc"][2]
    f = a[np.isfinite(a)]
    print(f"{nm:<16}{np.median(f):>10.1f}{np.percentile(f, 5):>9.1f}"
          f"{np.percentile(f, 95):>9.1f}{len(np.unique(f)):>18d}")
print("")
print("'distinct values' is the staircase, quantified: the threshold")
print("definition can only return model-level depths.")

In [ ]:
dfig.depth_map_grid(
    ["mld_threshold", "mld_interp", "mld_DI"], mv, CMAP_CFG,
    region=REGION, levels=("sfc",),
    row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle="Figure 2 — three MLD definitions on the same tile")
plt.show()

dfig.depth_pdf_grid(
    ["mld_threshold", "mld_interp", "mld_DI"], mv, CMAP_CFG,
    levels=("sfc",), row_labels=("whole tile",),
    suptitle=("Figure 3 — MLD distributions.  The threshold definition "
              "should show comb teeth at the model levels."))
plt.show()

Figure 3 is the test.  If `mld_threshold` shows **comb teeth** — spikes
at the model-level depths with gaps between — while `mld_interp` is
smooth, the staircase is confirmed and quantified.


## Section 4 — Does a continuous MLD actually fix the blotches?

The question that matters is not whether the MLD map looks nicer, but
whether a *field sampled at* the MLD comes out smoother.  Buoyancy is
the test case: it feeds `gradb2`, `R_ib` and `KE`.


In [ ]:
b = CF.buoyancy_of_field(ds_merge)


def field_at_depth_interp(field3d, depth2d):
    """Sample a 3D field at a continuous depth, linear in z.

    The counterpart of ``VH._extract_at_mld`` for a continuous MLD:
    interpolates between bracketing levels instead of snapping.

    Inputs
    ------
    field3d : xr.DataArray
        Lazy 3D field on tracer levels.
    depth2d : xr.DataArray
        2D target depth, m, positive downward.

    Outputs
    -------
    xr.DataArray
        2D field, dask-backed.

    Generated by LH and Claude.
    """
    zdim = VH._get_vertical_dim(field3d)

    def _interp(f, d):
        flat = f.reshape(-1, f.shape[-1]).astype(np.float64)
        dd = d.ravel().astype(np.float64)
        k = np.clip(np.searchsorted(Z, dd), 1, len(Z) - 1)
        z0, z1 = Z[k - 1], Z[k]
        w = np.where(z1 > z0, (dd - z0) / (z1 - z0), 0.0)
        rows = np.arange(flat.shape[0])
        out = (1 - w) * flat[rows, k - 1] + w * flat[rows, k]
        return out.reshape(d.shape).astype(np.float32)

    return xr.apply_ufunc(
        _interp, field3d, depth2d,
        input_core_dims=[[zdim], []], dask="parallelized",
        output_dtypes=[np.float32],
    )


b_snap = VH._extract_at_mld(b, mld, ds_merge)        # production
b_interp = field_at_depth_interp(b, mld_i)           # continuous

bv = dfig.pack_tile_levels(
    dict(zip(["b_at_mld_snapped", "b_at_mld_interp"],
             dask.compute(b_snap, b_interp))),
    XC, YC, edge_margin=0, land_mask=LAND, levels=("sfc",),
    verbose=False)

dfig.depth_map_grid(
    ["b_at_mld_snapped", "b_at_mld_interp"], bv, CMAP_CFG,
    region=REGION, levels=("sfc",),
    row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 4 — buoyancy at the MLD: production (snapped to "
              "a model level) vs continuous"))
plt.show()

In [ ]:
# Roughness: median |Laplacian|.  A staircase has large cell-to-cell
# jumps at the level boundaries, so it scores high; smooth physical
# structure scores low.
def roughness(a):
    """Median absolute discrete Laplacian, ignoring NaNs."""
    lap = (a[2:, 1:-1] + a[:-2, 1:-1] + a[1:-1, 2:] + a[1:-1, :-2]
           - 4 * a[1:-1, 1:-1])
    return np.nanmedian(np.abs(lap))


print(f"{'field':<22}{'roughness':>12}{'vs production':>16}")
print("-" * 50)
r0 = roughness(bv["b_at_mld_snapped"]["sfc"][2])
for nm in ("b_at_mld_snapped", "b_at_mld_interp"):
    r = roughness(bv[nm]["sfc"][2])
    print(f"{nm:<22}{r:>12.3e}{r / r0:>15.2f}x")
print("")
print("If the continuous version is materially smoother, the staircase")
print("was the dominant source of blotchiness in every _mld channel.")

## Section 5 — Decision

Fill in from the numbers above.  The options, in increasing order of
disruption:

1. **Change nothing.**  Document that `_mld` is quantised to model
   levels and should be read as such.  Correct, and free.
2. **Continuous MLD, snapped extraction.**  Make `mixed_layer_depth`
   interpolate to the exact threshold crossing, keep `_extract_at_mld`
   as-is.  This makes the MLD *channel* smooth but leaves every
   `_mld` field still sampling a model level — probably the worst of
   both, since the MLD channel would no longer match the level the
   other channels are drawn from.
3. **Continuous MLD and interpolated extraction.**  Both together, as
   in Section 4.  Internally consistent, and the only option that
   actually smooths the `_mld` channels.  Costs a vertical
   interpolation per field and changes every `_mld` output in the
   global store.
4. **A different criterion entirely** (N²-max, DI).  Section 3 compares
   these; they answer a different question and are not drop-in
   replacements.

Note that option 3 is not free of artifacts either: interpolating in z
near a sharp pycnocline is its own approximation, and it will smooth
real structure along with the staircase.  The question is which error
we would rather have, and that is a judgement call for LH, not
something the numbers settle on their own.

---

### Cross-references

- **The vertical stencil**, a different mechanism with similar
  symptoms — `vertical_gradients.ipynb`.
- **N² and the MLD channels themselves** —
  `depth_fields/stratification.ipynb`.
- **The fields most affected by a blotchy `_mld`** —
  `depth_fields/frontal_structure.ipynb`,
  `depth_fields/mixing_parameters.ipynb`.
